# Airbnb NYC Visualization 1

This notebook builds the borough average price chart from the cleaned dataset.

In [1]:
import pandas as pd
import altair as alt

# Constants
CSV_PATH = "AB_NYC_2019_clean_5000.csv"
OUTPUT_PATH = "viz1_from_notebook.html"
BAR_COLOR = "#e0754a"
BOROUGH_ORDER = ["Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"]
ROOM_OPTIONS = ["All", "Entire home/apt", "Private room", "Shared room"]

df = pd.read_csv(CSV_PATH)

# Compute avg price per borough and room type
agg = (
    df.groupby(["neighbourhood_group", "room_type"])["price"]
    .mean()
    .reset_index()
    .rename(columns={"neighbourhood_group": "borough", "price": "avg_price"})
)

# Add an "All" group
all_avg = (
    df.groupby("neighbourhood_group")["price"]
    .mean()
    .reset_index()
    .rename(columns={"neighbourhood_group": "borough", "price": "avg_price"})
)
all_avg["room_type"] = "All"

agg = pd.concat([agg, all_avg], ignore_index=True)

# Dropdown selection
room_dropdown = alt.binding_select(options=ROOM_OPTIONS, name="Room Type: ")
room_select = alt.param(name="room_type", value="All", bind=room_dropdown)

chart = (
    alt.Chart(agg)
    .mark_bar(color=BAR_COLOR)
    .encode(
        x=alt.X("borough:N", sort=BOROUGH_ORDER, title="NYC Borough"),
        y=alt.Y("avg_price:Q", title="Average Price (USD)"),
        tooltip=[
            alt.Tooltip("borough:N", title="Borough"),
            alt.Tooltip("avg_price:Q", title="Average Price", format="$.2f"),
        ],
    )
    .transform_filter(alt.datum.room_type == room_select)
    .add_params(room_select)
    .properties(
        title="Average Nightly Price by Borough",
        width=500,
        height=350,
    )
)

chart.save(OUTPUT_PATH)
chart

alt.Chart(...)